In [1]:
# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import warnings
import sys
import os

warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath(".."))

from functions.tools import configure_notebook_display, load_dataset, save_dataset

configure_notebook_display()

In [2]:
# ============================================================
# LOADING PROCESSED DATASETS
# ============================================================

trusted_readings = load_dataset(
    "../data/trusted_readings.csv",
    date_columns=["date"])

processed_metadata = load_dataset(
    "../data/processed_metadata.csv",
    date_columns=["sowing_date"])

Dataset loaded from: ../data/trusted_readings.csv
Dataset loaded from: ../data/processed_metadata.csv


In [3]:
# ============================================================
# DATE PARSING
# ============================================================

trusted_readings["date"] = pd.to_datetime(
    trusted_readings["date"])

processed_metadata["sowing_date"] = pd.to_datetime(
    processed_metadata["sowing_date"])

In [4]:
# ============================================================
# METADATA - READING MERGE
# ============================================================

merged_df = processed_metadata.merge(
    trusted_readings,
    on="parcel_id",
    how="left")

print("Metadata-anchor merge complete.")
print("Merged dataset shape:", merged_df.shape)

Metadata-anchor merge complete.
Merged dataset shape: (3410, 12)


The metadata dataset is being used as the main table while attaching sensor readings onto valid parcel entities.

Valid parcels are preserved even if they contain no readings, while rogue operational records remain excluded from the analytical layer.

In [5]:
# ============================================================
# STRUCTURALLY VALID PARCELS WITH NO READINGS
# ============================================================

missing_reading_parcels = merged_df[
    merged_df["date"].isna()
    ]

print("Parcels with metadata but no readings:")
print(missing_reading_parcels["parcel_id"].nunique())

display(
    missing_reading_parcels[
        [
            "parcel_id",
            "crop_type",
            "mill_id"
        ]
    ].drop_duplicates())

Parcels with metadata but no readings:
3


,parcel_id,crop_type,mill_id
3407,PARCEL_050,soybean,MILL_SOUTH
3408,PARCEL_051,sugarcane,MILL_NORTH
3409,PARCEL_052,soybean,MILL_WEST


`PARCEL_050`, `PARCEL_051`, `PARCEL_052` these parcels exist in metadata but contain no sensor readings are being isolated for operational visibility.

In [6]:
# ============================================================
# CREATING ANALYSIS ELIGIBLE DATASET
# ============================================================

analysis_df = merged_df[
    merged_df["date"].notna()
    ].copy()

print("Analysis eligible dataset shape:")
print(analysis_df.shape)

Analysis eligible dataset shape:
(3407, 12)


Rows containing valid reading activity are being separated into a dedicated analysis dataset.

The output confirms that the downstream analysis is now restricted to parcels with actual temporal observations while still preserving inactive parcels separately.

In [7]:
# ============================================
# Saving cleaned analysis Dataset
# ============================================

save_dataset(analysis_df, "../data/cleaned_parcel_timeseries.csv")

Dataset saved at: ../data/cleaned_parcel_timeseries.csv


In [8]:
# ============================================================
# CREATING RELATIVE DAY FEATURE
# ============================================================

analysis_df["days_from_sowing"] = (
    analysis_df["date"] -
    analysis_df["sowing_date"]
    ).dt.days

display(
    analysis_df[
        [
            "parcel_id",
            "date",
            "sowing_date",
            "days_from_sowing"
        ]
    ].head())

,parcel_id,date,sowing_date,days_from_sowing
0,PARCEL_001,2026-01-16,2026-02-10,-25
1,PARCEL_001,2026-01-26,2026-02-10,-15
2,PARCEL_001,2026-04-23,2026-02-10,72
3,PARCEL_001,2026-05-15,2026-02-10,94
4,PARCEL_001,2026-01-22,2026-02-10,-19


A sowing relative temporal feature is being created by measuring the distance between each reading date and its corresponding sowing date.



In [9]:
# ============================================================
# CREATING PRE-SOWING WINDOW
# ============================================================

before_sowing = analysis_df[
    (analysis_df["days_from_sowing"] >= -30) &
    (analysis_df["days_from_sowing"] <= -1)
    ].copy()

print("Before-sowing records:")
print(before_sowing.shape[0])

Before-sowing records:
367


Readings occurring within the 30 days before sowing are being isolated into a pre-sowing window.

In [10]:
# ============================================================
# CREATING POST-SOWING WINDOW
# ============================================================

after_sowing = analysis_df[
    (analysis_df["days_from_sowing"] >= 1) &
    (analysis_df["days_from_sowing"] <= 30)
    ].copy()

print("After-sowing records:")
print(after_sowing.shape[0])

After-sowing records:
553


Readings occurring within the 30 days after sowing are being isolated into a post-sowing window.

The output indicates stronger observation density after sowing, which could reflect increased operational monitoring during active crop development periods.

In [11]:
# ============================================================
# EXCLUDING CORRUPTED RECORDS
# ============================================================

before_sowing = before_sowing[
    before_sowing["observation_quality"] != "CORRUPTED"]

after_sowing = after_sowing[
    after_sowing["observation_quality"] != "CORRUPTED"]

Rows marked as CORRUPTED (invalid NDVI) are being excluded only from the sowing-window calculations. This is done to make sure invalid NDVI is no longer influencing crop-level vegetation calculations.

Corrupted observations are excluded here because their NDVI values fall outside the valid biological range. Degraded and unknown records are still preserved since operational uncertainty does not automatically mean the vegetation signal itself is invalid.

In [12]:
# ============================================================
# MEAN NDVI BEFORE SOWING
# ============================================================

before_summary = (
    before_sowing
    .groupby("crop_type")
    .agg(
        mean_ndvi_before=("ndvi_value", "mean"),
        n_parcels_before=("parcel_id", "nunique"),
        readings_before=("parcel_id", "count")
    )
    .reset_index())

display(before_summary)

,crop_type,mean_ndvi_before,n_parcels_before,readings_before
0,soybean,0.215545,4,77
1,sugarcane,0.230894,19,226
2,wheat,0.215500,2,50


Average NDVI values are being calculated for each crop type during the 30 days leading up to sowing.

In [13]:
# ============================================================
# MEAN NDVI AFTER SOWING
# ============================================================

after_summary = (
    after_sowing
    .groupby("crop_type")
    .agg(
        mean_ndvi_after=("ndvi_value", "mean"),
        n_parcels_after=("parcel_id", "nunique"),
        readings_after=("parcel_id", "count")
    )
    .reset_index())

display(after_summary)

,crop_type,mean_ndvi_after,n_parcels_after,readings_after
0,soybean,0.316259,4,81
1,sugarcane,0.355916,19,418
2,wheat,0.322548,2,42


Average NDVI values are being calculated for each crop type during the 30 days following sowing.

In [14]:
# ============================================================
# REQIURED OUTPUT AS ASKED IN THR PROBLEM
# ============================================================

final_summary = before_summary.merge(
    after_summary,
    on="crop_type",
    how="outer"
)

final_summary = final_summary.rename(
    columns={"n_parcels_after": "n_parcels"}
)

final_summary = final_summary[
    [
        "crop_type",
        "mean_ndvi_before",
        "mean_ndvi_after",
        "n_parcels"
    ]
]

display(final_summary)

,crop_type,mean_ndvi_before,mean_ndvi_after,n_parcels
0,soybean,0.215545,0.316259,4
1,sugarcane,0.230894,0.355916,19
2,wheat,0.215500,0.322548,2


All crop types show higher average NDVI values after sowing compared to the period before sowing. Since NDVI measures vegetation health and density through light reflectance, the increase after sowing aligns with expected crop growth behavior as vegetation becomes more active and developed. Sugarcane shows the strongest post-sowing NDVI levels, while soybean and wheat also display moderate increases, suggesting that most parcels follow biologically plausible growth patterns after planting.

Final Dataset Layers

| Dataset | Purpose |
|---|---|
| trusted_readings.csv | Cleaned and valid observations |
| rogue_readings.csv | Metadata inconsistent parcel records |
| processed_metadata.csv | Standardized parcel metadata |
| cleaned_parcel_timeseries.csv | Cleaned and Merged Dataset asked in the brief |
| analysis_dataset.csv | Final analysis dataset |
